In [48]:
!pip install -U yfinance -q

In [49]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


def data(Ticker  , start , end) :
  x = yf.download(Ticker  , start , end , multi_level_index=False)
  x = x.ffill()
  return x

df1 = data('HDFCBANK.NS' , '2020-01-01' , date.today())
df2 = data('TCS.NS' , '2020-01-01' , date.today())
df3 = data('RELIANCE.NS' , '2020-01-01' , date.today())
df4 = data('SUNPHARMA.NS' , '2020-01-01' , date.today())
df5 = data('HINDUNILVR.NS' , '2020-01-01' , date.today())
df6 = data('BHARTIARTL.NS' , '2020-01-01' , date.today())
df7 = data('M&M.NS' , '2020-01-01' , date.today())
df8 = data('LT.NS' , '2020-01-01' , date.today())
df_b = data('^NSEI' , '2020-01-01' , date.today())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [50]:
#backtest

def backtest(signal , close, fee=0.0025 , alloc = 0.25 , capital = 1000000) :
  shares = ( (capital * alloc) / close.shift(1) ).fillna(0).astype(int)
  daily_pnl = close.diff() * signal.shift(1) * shares
  position = signal.diff().fillna(0)
  daily_fees = fee * shares * position.shift(1).abs() * close.shift(1)
  equity = capital + (daily_pnl - daily_fees).cumsum()
  return equity

In [51]:

#Moving Average

capital = 1000000
def Moving_Average(df , wind_s1 , wind_s2) :
  SMA_fast = df['Close'].rolling(window = wind_s1).mean()
  SMA_slow = df['Close'].rolling(window = wind_s2).mean()

  Signal = (SMA_fast > SMA_slow).astype(int)

  return Signal

#Breakout Strategy

def Breakout_strategy(df , window_sz) :
  wind_high = df['Close'].shift(1).rolling(window = window_sz).max()
  wind_low = df['Close'].shift(1).rolling(window = window_sz).min()

  raw_signal = np.select(
      [df['Close'] > wind_high , df['Close'] < wind_low] ,
      [1,0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal, index=df.index).ffill().fillna(0)

  return signal

#RSI mean reversion Strategy

def RSI_MR(df , wind_sz) :

  delta = df['Close'].diff()

  gain = delta.clip(lower = 0)
  loss = -delta.clip(upper = 0)

  avg_gain = gain.rolling(window = wind_sz).mean()
  avg_loss = loss.rolling(window = wind_sz).mean()

  rs = avg_gain / avg_loss

  rs_val = 100 - (100 / (1 +  rs))

  raw_signal = np.select(
      [(rs_val.shift(1) < 30) & (rs_val >= 30) , (rs_val.shift(1) < 50) & (rs_val >= 50)] ,
      [1 , 0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal , index = df.index).ffill().fillna(0)

  return signal

In [52]:
stocks = {
    "HDFCBANK": df1,
    "TCS": df2,
    "RELIANCE": df3,
    "SUNPHARMA": df4,
    "HINDUNILVR": df5,
    "BHARTIARTL": df6,
    "M&M": df7,
    "LT": df8,
}

results = {}
signals = {}

for stock , df in stocks.items() :
    sigs = {"MA": Moving_Average(df, 10, 20),
            "Breakout": Breakout_strategy(df, 20),
            "RSI": RSI_MR(df, 14)}
    signals[stock] = sigs
    results[stock] = pd.DataFrame(
        {s : backtest(sig , df['Close']) for s , sig in sigs.items()}
    )

net_results = pd.concat(results , axis = 1)
# net_results
# print(signals['HDFCBANK']['MA'])

In [53]:
def performance_report(signal, close) :
  # print(close)
  fee = 0.0025

  strategy_ret = signal.shift(1) * close.pct_change() - fee * signal.diff().shift(1).abs().fillna(0)
  cumm_strategy_ret = (1 + strategy_ret).cumprod()


  N = signal.shape[0]
  rf = 0.065
  daily_rf = rf/252

  ann_ret = (cumm_strategy_ret.iloc[-1]) ** (252 / N) - 1
  ann_vol = strategy_ret.std() * np.sqrt(252)
  sharpe =  (ann_ret - rf) / (ann_vol)

  downside_diff = np.minimum(0, strategy_ret - daily_rf)
  downside_dev = np.sqrt((downside_diff ** 2).mean()) * np.sqrt(252)

  sortino = (ann_ret - rf) / downside_dev

  #drawdown analysis:
  dd = (cumm_strategy_ret - cumm_strategy_ret.cummax()) / cumm_strategy_ret.cummax()
  max_dd = dd.min()
  max_dd_date = dd.idxmin()

  avg_dd = dd.mean()

  running_max = cumm_strategy_ret.cummax()

  trough_date = dd.idxmin()
  peak_value = running_max.loc[trough_date]

  # 2. Slice strictly from the trough forward and filter for full recovery
  post_trough = cumm_strategy_ret.loc[trough_date:]
  recovered = post_trough[post_trough >= peak_value]

  # print(recovered)

  if recovered.empty:
    recovery_days = np.nan
  else:
    recovery_date = recovered.index[0]
    recovery_days = len(post_trough.loc[:recovered.index[0]]) - 1

  pos = signal.shift(1).fillna(0)
  days_invested = pos.cumsum().iloc[-1]
  exposure = 1 - pos.mean()   #fraction of time invested
  cash_pct = 1 - exposure     #fraction of time in cash

  return {
      "Cumulative Return": cumm_strategy_ret.iloc[-1] - 1,
      "Annual Return": ann_ret,
      "Annual Volatility": ann_vol,
      "Sharpe": sharpe,
      "Sortino": sortino,
      "Max Drawdown": max_dd,
      "Max DD Date": max_dd_date,
      "Avg Drawdown": avg_dd,
      "Recovery Days": recovery_days,
  }

performance_report(signals['RELIANCE']['MA'], df3['Close'])

{'Cumulative Return': np.float64(0.3625930450069843),
 'Annual Return': np.float64(0.046761540108038124),
 'Annual Volatility': np.float64(0.18644832448366058),
 'Sharpe': np.float64(-0.09782045476927956),
 'Sortino': np.float64(-0.14938966669202708),
 'Max Drawdown': -0.39563676261881836,
 'Max DD Date': Timestamp('2026-10-28 00:00:00'),
 'Avg Drawdown': np.float64(-0.1544367395131703),
 'Recovery Days': nan}